# Reading Dolos with J, R, and J++

A hands-on introduction to six matched workspace lenses: three for **Dolos-1.5B**, three for **Qwen2.5-1.5B-Instruct**. We will inspect the top words that each lens reads out at intermediate layers, compare three estimators, and examine the parent/fine-tune pair.

**What these scores are not:** thoughts, guaranteed semantic features, probabilities of an intention, or evidence of deception. The lenses are linear transports of residual activations to the model's unembedding. The examples below are probes, not training or statistical evaluation.

Inspired by the intermediate-variable examples in [Verbalizable Representations Form a Global Workspace in Language Models](https://transformer-circuits.pub/2026/workspace/index.html) and the [J++ quickstart](https://github.com/safety-research/jpp_lens/blob/main/scripts/quickstart.py). We don't assume that a small model faithfully solves each example.

## 0. Setup — Google Colab or local

**Colab:** Choose **Runtime → Change runtime type → T4 GPU** (or better). On a new session, run the first code cell once. It installs the official J++ package and deliberately stops. Choose **Runtime → Restart session** (not factory reset), then **Run all**. The restart matters: installing/upgrading NumPy within an already running Colab kernel can leave a mixture of old loaded binary extensions and new Python packages, producing errors such as `_blas_supports_fpe` / `GenerationMixin`.

**Local / training pod:** Set `SOURCE = "local"` in the configuration cell. The old `/workspace/dolos` paths remain supported. An existing environment can skip installation by using a separate initialized Python environment. Python **3.12+** is required.

The models load **sequentially** to conserve VRAM. Notebook examples are qualitative, not formal held-out evaluation.

In [ ]:
# Run FIRST on a fresh Colab runtime. Do not import torch, pandas or numpy before this.
import importlib.metadata as metadata
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 12):
    raise RuntimeError("Official J++ requires Python 3.12+; choose a newer Colab runtime.")

checkout = Path("/content/jpp_lens")
marker = Path("/content/.dolos_reader_installed")
if not marker.exists():
    if not (checkout / "src/workspace_lens").is_dir():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/safety-research/jpp_lens.git",
                        str(checkout)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(checkout)], check=True)
    marker.write_text("installed\n")
    print("Dependencies installed. IMPORTANT: Runtime → Restart session, then Run all.")
    # Stop immediately: pip may have replaced NumPy while Colab has its old binary loaded.
    raise RuntimeError("One-time setup complete. Restart the Colab session, then Run all.")
else:
    # /content survives kernel restarts within the same Colab session.
    sys.path.insert(0, str(checkout / "src"))
    from workspace_lens import get_hf_model
    from workspace_lens.utils import load_lens_file, check_model_matches_config
    print("J++ reader OK; NumPy", metadata.version("numpy"),
          "Transformers", metadata.version("transformers"))

In [ ]:
# Configuration: edit SOURCE to "local" only when running on the training pod.
from pathlib import Path
import gc
import torch
import pandas as pd
from IPython.display import display, Markdown

SOURCE = "hub"
HF_REPO = "bubskee/dolos-qwen25-workspace-lenses"
ROOT = Path("/workspace/dolos")
LAYERS = [0, 4, 8, 12, 16, 20, 24, 26]
TOP_K = 6

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU required. In Colab: Runtime → Change runtime type → T4 GPU.")
print("Torch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0), "source:", SOURCE)

In [ ]:
if SOURCE == "local":
    model_dirs = {"dolos": ROOT / "models/dolos", "parent": ROOT / "models/parent"}
    lens_files = {
        m: {k: ROOT / "artifacts" / f"{prefix}_{k}_n64.pt" for k in ("j", "r", "jpp")}
        for m, prefix in (("dolos", "dolos"), ("parent", "parent"))
    }
elif SOURCE == "hub":
    from huggingface_hub import HfApi, snapshot_download

    # Ask the Hub for its actual published file list, not the pod layout.
    published_files = set(HfApi().list_repo_files(repo_id=HF_REPO, repo_type="model"))
    required_files = {
        "models/dolos/config.json",
        *(f"lenses/{m}/{k}/lens.pt" for m in ("dolos", "parent") for k in ("j", "r", "jpp")),
    }
    absent = sorted(required_files - published_files)
    if absent:
        raise FileNotFoundError(f"Hub release is missing required files: {absent}")
    print("Verified published Hub layout:")
    for name in sorted(f for f in published_files if f.startswith(("models/", "lenses/"))):
        print(" ", name)

    # Download exactly the files needed for this walkthrough, not provenance/logs.
    snap = Path(snapshot_download(
        repo_id=HF_REPO,
        repo_type="model",
        allow_patterns=["models/dolos/**", "lenses/**"],
    ))
    model_dirs = {
        "dolos": snap / "models/dolos",
        "parent": "Qwen/Qwen2.5-1.5B-Instruct",
    }
    lens_files = {
        m: {k: snap / "lenses" / m / k / "lens.pt" for k in ("j", "r", "jpp")}
        for m in ("dolos", "parent")
    }
else:
    raise ValueError("SOURCE must be 'local' or 'hub'")

missing = [f"{m}/{k}: {p}" for m, lenses in lens_files.items()
           for k, p in lenses.items() if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing published lens artifacts:\n" + "\n".join(missing))
if SOURCE == "hub" and not (model_dirs["dolos"] / "config.json").is_file():
    raise FileNotFoundError(f"Missing Dolos config in {model_dirs['dolos']}; check release layout.")
print("Found all six lens files.")
# Release layout preflight: confirm exact files without downloading research logs.
print("Model paths:", {k: str(v) for k, v in model_dirs.items()})
print("Lens paths:", {m: {k: str(v) for k, v in ks.items()} for m, ks in lens_files.items()})


## 1. Three examples

The first two prompts evoke an **intermediate concept** (heart, Italy) that is not the desired final answer (four, euro). The third does likewise for a spider and its number of legs. These are explanatory examples, not evidence that the model actually computes via these concepts.

In [ ]:
EXAMPLES = {
    "Heart → chambers": "Fact: In humans, the organ that pumps blood through the body has this many chambers: ",
    "Italy → currency": "Fact: The currency used in the country shaped like a boot is",
    "Spider → legs": "Question: How many legs does the animal that spins webs have? Answer:",
}
EXAMPLES

## 2. Load one model and its three lenses

The reader checks lens/config compatibility before applying it. **Use the exact matching checkpoint**; parent and Dolos lenses aren't interchangeable. Each model is loaded once, and its lenses are read one by one to reduce RAM usage.

**Published-path note:** The lens files record their original training-pod paths. The notebook checks those paths and checkpoint labels before rebinding model paths in memory for the Hugging Face copies. This is not a hash verification of model weights.


In [ ]:
TRAINING_MODEL_PATHS = {
    "dolos": "/workspace/dolos/models/dolos",
    "parent": "/workspace/dolos/models/parent",
}
MODEL_HUB_IDENTITIES = {
    "dolos": "bubskee/dolos-qwen25-workspace-lenses",
    "parent": "Qwen/Qwen2.5-1.5B-Instruct",
}

def load_bundle(model_key):
    if model_key not in model_dirs:
        raise ValueError(f"Unknown model key: {model_key!r}")
    model_path = str(model_dirs[model_key])
    model = get_hf_model(model_path, attn_implementation="sdpa")
    from workspace_lens.utils import hf_model_name_of
    actual_name = hf_model_name_of(model)
    if actual_name != model_path:
        raise ValueError(f"Loaded unexpected model identity {actual_name!r}; expected {model_path!r}")
    # Load lenses lazily during inspect(): holding all three dense Jacobian
    # dictionaries alongside model weights can exhaust Colab's system RAM.
    return model, model_key

def load_one_lens(model, model_key, kind):
    from workspace_lens.utils import hf_model_name_of
    model_path = str(model_dirs[model_key])
    actual_name = hf_model_name_of(model)
    p = lens_files[model_key][kind]
    lens = load_lens_file(str(p), hf_model_name=model_path)
    if SOURCE == "hub":
        original_path = TRAINING_MODEL_PATHS[model_key]
        if lens.config.hf_model_name != original_path:
            raise ValueError(f"{model_key}/{kind} lens fitted on {lens.config.hf_model_name!r}, expected {original_path!r}")
        checkpoint_kind = {"j": "j", "r": "lrp", "jpp": "jpp"}[kind]
        if not lens.config.checkpoint_name.startswith(f"{model_key}_{checkpoint_kind}_"):
            raise ValueError(f"Unexpected checkpoint {lens.config.checkpoint_name!r}")
        lens.config.hf_model_name = actual_name
    check_model_matches_config(model, lens.config)
    assert set(LAYERS).issubset(set(lens.source_layers)), (kind, lens.source_layers)
    return lens


def unload(model, lenses):
    del model, lenses
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

model, lenses = load_bundle("dolos")
print("Loaded Dolos model; lenses will be streamed one at a time.")

## 3. Read out top tokens by layer

The score is obtained by transporting the residual through a **fixed lens map** and applying the model's final unembedding. Rankings below use a simple transparent filter (exclude tokens containing no letters or digits); this is a *tutorial convenience*, not a reproduction of the published evaluation's candidate-preserving readout filter.

In [ ]:
def meaningful_tokens(model):
    # Filter token ids only for display. To compare published recall metrics,
    # use the official evaluation code with its candidate-preserving filter.
    tok = model.tokenizer
    n = model.vocab_size if hasattr(model, "vocab_size") else len(tok)
    excluded = []
    for i in range(n):
        s = tok.decode([i], skip_special_tokens=False)
        if not any(ch.isalnum() for ch in s):
            excluded.append(i)
    return excluded

def inspect(model, lenses, prompt, layers=LAYERS, top_k=TOP_K, excluded=None):
    excluded = meaningful_tokens(model) if excluded is None else excluded
    rows = []
    for kind in ("j", "r", "jpp"):
        lens = load_one_lens(model, lenses, kind)
        lens_logits, model_logits, _ = lens.apply(
            model, prompt, layers=layers, token_positions_for_residuals=[-1]
        )
        for layer in layers:
            scores = lens_logits[layer][0].float().clone()
            scores[excluded] = -float("inf")
            ids = scores.topk(top_k).indices.tolist()
            rows.append({"lens": kind.upper(), "layer": layer,
                         "top tokens": " · ".join(repr(model.tokenizer.decode([i])) for i in ids)})
        del lens, lens_logits
        gc.collect()
    pred_ids = model_logits[0].float().topk(top_k).indices.tolist()
    print("Prompt:", prompt)
    print("Model next-token candidates:", [model.tokenizer.decode([i]) for i in pred_ids])
    return pd.DataFrame(rows)

excluded = meaningful_tokens(model)
display(inspect(model, lenses, EXAMPLES["Heart → chambers"], excluded=excluded))

In [ ]:
for title in ("Italy → currency", "Spider → legs"):
    display(Markdown(f"### {title}"))
    display(inspect(model, lenses, EXAMPLES[title], excluded=excluded))

## 4. Why a lens result isn't a behavioral result

Look for an intermediate word appearing at earlier layers in J++ than J or R. But don't cherry-pick: a word like `heart` can be highly ranked because of word associations, even if it isn't causally used to arrive at the answer. Likewise, a strong next-token prediction isn't proof the intermediate was explicitly represented. The six-lens release contains held-out aggregate evaluations; this notebook instead presents qualitative readouts.

## 5. Compare to the parent checkpoint

Repeat the same prompt using parent lenses on the parent model. The prompt stays fixed, but the fitted maps and weights change. This produces **descriptive** differences, not a controlled causal account of fine-tuning. Note that the formal parent and Dolos held-out evaluations use separately correctness-filtered populations.

In [ ]:
# Compare the same prompt, loading one model at a time.
dolos_example = inspect(model, lenses, EXAMPLES["Heart → chambers"], excluded=excluded)
del model, lenses, excluded
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

parent_model, parent_lenses = load_bundle("parent")
parent_excluded = meaningful_tokens(parent_model)
parent_example = inspect(parent_model, parent_lenses, EXAMPLES["Heart → chambers"], excluded=parent_excluded)
display(Markdown("**Dolos**"))
display(dolos_example)
display(Markdown("**Parent Qwen2.5**"))
display(parent_example)

## 6. Try your own prompt

A useful probe makes the *bridging variable* distinguishable from the final answer. Try changing the phrasing, add a distractor, or compare two prompts that differ in one substantive fact. Do not infer hidden objectives from isolated top-token lists.

In [ ]:
MY_PROMPT = "Fact: The number of legs on the animal that spins webs is"
display(inspect(parent_model, parent_lenses, MY_PROMPT, excluded=parent_excluded))

## 7. Further reading

- [Original workspace/J-space paper](https://transformer-circuits.pub/2026/workspace/index.html)
- [J++ code and explanations](https://github.com/safety-research/jpp_lens)
- [Dolos six-lens methodology](https://github.com/bfaber-centaur/dolos-lenses/blob/main/docs/DESIGN.md)

**Reproducibility:** For numerical scores and disjoint calibration/held-out samples, use the release metadata and the official `jpp_cli.py evaluate` procedure; the tutorial's token display filter is not the paper's formal scoring protocol.